In [ ]:
from pathlib import Path
import gc
import pickle

import numpy as np
import pandas as pd
import torch
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion


cwd = Path.cwd().resolve()


def find_release_root(start: Path) -> Path:
    sentinel_dirs = ("Step0_Data", "Step7_ModelTraining")
    nested_release_dirs = ("Global-solid-waste-prediction",)
    for candidate in [start, *start.parents]:
        if all((candidate / name).is_dir() for name in sentinel_dirs):
            return candidate
        for release_dir in nested_release_dirs:
            release_candidate = candidate / release_dir
            if all((release_candidate / name).is_dir() for name in sentinel_dirs):
                return release_candidate
    raise FileNotFoundError("Could not locate GitHub release root from the current working directory")


release_root = find_release_root(cwd)
STEP_ROOT = release_root / "Step7_ModelTraining"
OUTPUT_DIR = STEP_ROOT / "2_Results"

MIXED_MODEL_ID = "tabpfn_v26_mixed_finalize_core10_route"
WASTE_BY_MODEL_ID = "tabpfn_v26_waste_by_finalize_core10"
WASTE_TYPES = ["AW", "CDW", "IW", "MSW"]
MIXED_ROUTE_FEATURES = ["WasteRoute_AW", "WasteRoute_CDW", "WasteRoute_IW", "WasteRoute_MSW"]
TARGET_COL = "Target_Log"
TABPFN_N_ESTIMATORS = 8
TABPFN_SUBSAMPLE_SAMPLES = 2000
SEED = 42
BATCH_SIZE = 512
PREDICT_PROGRESS_EVERY = 4
HIGH_IMPACT_TOP_N = 80

FULL_LONG_TABLE_PATH = OUTPUT_DIR / "0_full_long_table.csv"
RAW_CONTRACT_PATH = OUTPUT_DIR / "0_feature_contract_raw.csv"
WASTE_BY_PREDICTIONS_PATH = OUTPUT_DIR / "1_predictions_all_wastes_long.csv"

MIXED_RAW_CONTRACT_PATH = OUTPUT_DIR / "2_mixed_feature_contract_raw.csv"
MIXED_PROCESSED_CONTRACT_PATH = OUTPUT_DIR / "2_mixed_feature_contract_processed.csv"
MIXED_PIPELINE_PATH = OUTPUT_DIR / "2_mixed_preprocessing_pipeline.pkl"
MIXED_MODEL_PATH = OUTPUT_DIR / "2_mixed_finalize_model.pkl"
MIXED_PREDICTIONS_PATH = OUTPUT_DIR / "2_mixed_predictions_all_wastes_long.csv"
ROUTE_COMPARISON_OVERALL_PATH = OUTPUT_DIR / "2_mixed_vs_wasteby_metrics_overall.csv"
ROUTE_COMPARISON_BY_WASTE_PATH = OUTPUT_DIR / "2_mixed_vs_wasteby_metrics_by_waste.csv"
HIGH_IMPACT_PATH = OUTPUT_DIR / "2_mixed_vs_wasteby_high_impact_country_waste.csv"
MSW_DIAGNOSTICS_PATH = OUTPUT_DIR / "2_msw_cross_waste_contamination_diagnostics.csv"

CONTEXT_COLUMNS = ["Country Code", "Country Name", "Year", "IncomeGroup", "Region", "WasteFlag"]
PREDICTION_COLUMNS = [
    *CONTEXT_COLUMNS,
    "Label_Raw_Observed",
    "Label_Log_Observed",
    "Mixed_Predicted_Raw",
    "Mixed_Predicted_Log",
    "Mixed_Final_Raw",
    "Mixed_Final_Log",
    "Mixed_Source",
    "HasObservedLabel",
]


class MixedPreprocessor:
    def __init__(self, columns: list[str], medians: dict[str, float], means: dict[str, float], stds: dict[str, float]):
        self.columns = columns
        self.medians = medians
        self.means = means
        self.stds = stds


def fit_mixed_preprocessor(frame: pd.DataFrame, feature_columns: list[str]) -> MixedPreprocessor:
    medians = {}
    means = {}
    stds = {}
    for column in feature_columns:
        series = pd.to_numeric(frame[column], errors="coerce").astype(float)
        median = float(series.median())
        if not np.isfinite(median):
            raise RuntimeError(f"Cannot fit mixed preprocessor because {column} has no finite median")
        filled = series.fillna(median)
        mean = float(filled.mean())
        std = float(filled.std(ddof=0))
        if not np.isfinite(std) or std == 0.0:
            std = 1.0
        medians[column] = median
        means[column] = mean
        stds[column] = std
    return MixedPreprocessor(columns=feature_columns.copy(), medians=medians, means=means, stds=stds)


def transform_with_mixed_preprocessor(preprocessor: MixedPreprocessor, frame: pd.DataFrame) -> pd.DataFrame:
    missing = [column for column in preprocessor.columns if column not in frame.columns]
    if missing:
        raise RuntimeError(f"Missing mixed preprocessor columns: {missing}")
    out = pd.DataFrame(index=frame.index)
    for column in preprocessor.columns:
        series = pd.to_numeric(frame[column], errors="coerce").astype(float).fillna(preprocessor.medians[column])
        out[column] = (series - preprocessor.means[column]) / preprocessor.stds[column]
    if int(out.isna().sum().sum()) != 0:
        raise RuntimeError("Mixed preprocessor produced NaN values")
    return out.reset_index(drop=True)

OUTPUT_DIR


In [ ]:
def require_file(path: Path) -> Path:
    if not path.is_file():
        raise FileNotFoundError(f"Required file does not exist: {path}")
    return path


def resolve_model_version():
    if not hasattr(ModelVersion, "V2_6"):
        raise RuntimeError("TabPFN ModelVersion.V2_6 is required; version fallback is not allowed")
    return ModelVersion.V2_6


def add_waste_route_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    if "WasteFlag" not in out.columns:
        raise KeyError("WasteFlag column is required to build mixed route features")
    actual_wastes = sorted(out["WasteFlag"].dropna().astype(str).unique().tolist())
    if actual_wastes != WASTE_TYPES:
        raise RuntimeError(f"Unexpected waste flags: {actual_wastes}")
    for waste_type in WASTE_TYPES:
        out[f"WasteRoute_{waste_type}"] = (out["WasteFlag"].astype(str) == waste_type).astype(float)
    return out


def predict_in_batches(model, features: pd.DataFrame, batch_size: int, progress_every: int) -> np.ndarray:
    n_rows = len(features)
    if n_rows == 0:
        return np.asarray([], dtype=float)
    if batch_size <= 0 or batch_size >= n_rows:
        return np.asarray(model.predict(features), dtype=float).reshape(-1)
    predictions = []
    n_batches = (n_rows - 1) // batch_size + 1
    for batch_idx in range(n_batches):
        start = batch_idx * batch_size
        end = min((batch_idx + 1) * batch_size, n_rows)
        batch_features = features.iloc[start:end]
        batch_pred = np.asarray(model.predict(batch_features), dtype=float).reshape(-1)
        predictions.append(batch_pred)
        if progress_every and (((batch_idx + 1) % progress_every == 0) or (batch_idx + 1 == n_batches)):
            print(f"Prediction batch {batch_idx + 1}/{n_batches} ({end}/{n_rows} rows)")
    return np.concatenate(predictions, axis=0)


def compute_metric_row(route_name: str, frame: pd.DataFrame, pred_log_col: str) -> dict[str, float | int | str]:
    observed = frame[frame["HasObservedLabel"].astype(bool)].copy()
    y_true_log = pd.to_numeric(observed["Label_Log_Observed"], errors="coerce").to_numpy(dtype=float)
    y_pred_log = pd.to_numeric(observed[pred_log_col], errors="coerce").to_numpy(dtype=float)
    valid_mask = np.isfinite(y_true_log) & np.isfinite(y_pred_log)
    y_true_log = y_true_log[valid_mask]
    y_pred_log = y_pred_log[valid_mask]
    y_true_raw = np.expm1(y_true_log)
    y_pred_raw = np.expm1(y_pred_log)
    eps = 1e-12
    return {
        "route": route_name,
        "rows": int(len(y_true_log)),
        "WAPE_log": float(np.abs(y_true_log - y_pred_log).sum() / max(np.abs(y_true_log).sum(), eps)),
        "R2_log": float(1.0 - ((y_true_log - y_pred_log) ** 2).sum() / max(((y_true_log - y_true_log.mean()) ** 2).sum(), eps)),
        "MAE_original": float(np.abs(y_true_raw - y_pred_raw).mean()),
        "Mean_ARE": float(np.mean(np.abs((y_pred_raw - y_true_raw) / np.maximum(y_true_raw, eps)))),
        "Median_ARE": float(np.median(np.abs((y_pred_raw - y_true_raw) / np.maximum(y_true_raw, eps)))),
    }


def build_route_switch_comparison(comparison_frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    observed = comparison_frame[comparison_frame["HasObservedLabel"].astype(bool)].copy()
    observed["Mixed_ARE"] = (observed["Mixed_Predicted_Raw"] - observed["Label_Raw_Observed"]).abs() / observed["Label_Raw_Observed"].abs()
    observed["WasteBy_ARE"] = (observed["WasteBy_Predicted_Raw"] - observed["Label_Raw_Observed"]).abs() / observed["Label_Raw_Observed"].abs()
    observed["Mixed_SRE"] = (observed["Mixed_Predicted_Raw"] - observed["Label_Raw_Observed"]) / observed["Label_Raw_Observed"]
    observed["WasteBy_SRE"] = (observed["WasteBy_Predicted_Raw"] - observed["Label_Raw_Observed"]) / observed["Label_Raw_Observed"]
    overall = pd.DataFrame([
        compute_metric_row("mixed_core10_route", comparison_frame, "Mixed_Predicted_Log"),
        compute_metric_row("waste_by_core10", comparison_frame, "WasteBy_Predicted_Log"),
    ])
    waste_rows = []
    for waste_type in WASTE_TYPES:
        waste_frame = comparison_frame[comparison_frame["WasteFlag"] == waste_type].copy()
        mixed_row = compute_metric_row("mixed_core10_route", waste_frame, "Mixed_Predicted_Log")
        waste_by_row = compute_metric_row("waste_by_core10", waste_frame, "WasteBy_Predicted_Log")
        mixed_row["WasteFlag"] = waste_type
        waste_by_row["WasteFlag"] = waste_type
        waste_rows.extend([mixed_row, waste_by_row])
    by_waste = pd.DataFrame(waste_rows)
    country_waste = (
        observed.groupby(["Country Code", "Country Name", "Region", "IncomeGroup", "WasteFlag"], as_index=False)
        .agg(
            rows=("Label_Raw_Observed", "size"),
            total_actual=("Label_Raw_Observed", "sum"),
            mean_actual=("Label_Raw_Observed", "mean"),
            mixed_mean_are=("Mixed_ARE", "mean"),
            waste_by_mean_are=("WasteBy_ARE", "mean"),
            mixed_median_are=("Mixed_ARE", "median"),
            waste_by_median_are=("WasteBy_ARE", "median"),
            mixed_mean_sre=("Mixed_SRE", "mean"),
            waste_by_mean_sre=("WasteBy_SRE", "mean"),
        )
    )
    country_waste["are_improvement_mixed_minus_waste_by"] = country_waste["mixed_mean_are"] - country_waste["waste_by_mean_are"]
    country_waste["waste_by_better"] = country_waste["waste_by_mean_are"] < country_waste["mixed_mean_are"]
    high_impact = country_waste.sort_values("total_actual", ascending=False).head(HIGH_IMPACT_TOP_N).reset_index(drop=True)
    return overall, by_waste, high_impact


def build_msw_contamination_diagnostics(comparison_frame: pd.DataFrame) -> pd.DataFrame:
    observed = comparison_frame[comparison_frame["HasObservedLabel"].astype(bool)].copy()
    observed["Mixed_ARE"] = (observed["Mixed_Predicted_Raw"] - observed["Label_Raw_Observed"]).abs() / observed["Label_Raw_Observed"].abs()
    observed["WasteBy_ARE"] = (observed["WasteBy_Predicted_Raw"] - observed["Label_Raw_Observed"]).abs() / observed["Label_Raw_Observed"].abs()
    observed["Mixed_SRE"] = (observed["Mixed_Predicted_Raw"] - observed["Label_Raw_Observed"]) / observed["Label_Raw_Observed"]
    observed["WasteBy_SRE"] = (observed["WasteBy_Predicted_Raw"] - observed["Label_Raw_Observed"]) / observed["Label_Raw_Observed"]
    rows = []
    for waste_type in WASTE_TYPES:
        part = observed[observed["WasteFlag"] == waste_type].copy()
        rows.append({
            "WasteFlag": waste_type,
            "rows": int(len(part)),
            "total_actual": float(part["Label_Raw_Observed"].sum()),
            "mixed_mean_are": float(part["Mixed_ARE"].mean()),
            "waste_by_mean_are": float(part["WasteBy_ARE"].mean()),
            "mixed_mean_sre": float(part["Mixed_SRE"].mean()),
            "waste_by_mean_sre": float(part["WasteBy_SRE"].mean()),
            "mixed_over_waste_by_are_ratio": float(part["Mixed_ARE"].mean() / max(part["WasteBy_ARE"].mean(), 1e-12)),
        })
    diagnostics = pd.DataFrame(rows)
    msw_mixed_are = float(diagnostics.loc[diagnostics["WasteFlag"] == "MSW", "mixed_mean_are"].iloc[0])
    msw_waste_by_are = float(diagnostics.loc[diagnostics["WasteFlag"] == "MSW", "waste_by_mean_are"].iloc[0])
    non_msw_mixed_are = float(diagnostics.loc[diagnostics["WasteFlag"] != "MSW", "mixed_mean_are"].mean())
    non_msw_waste_by_are = float(diagnostics.loc[diagnostics["WasteFlag"] != "MSW", "waste_by_mean_are"].mean())
    diagnostics["msw_vs_non_msw_mixed_are_gap"] = msw_mixed_are - non_msw_mixed_are
    diagnostics["msw_vs_non_msw_waste_by_are_gap"] = msw_waste_by_are - non_msw_waste_by_are
    return diagnostics

raw_feature_columns = pd.read_csv(require_file(RAW_CONTRACT_PATH))["feature"].astype(str).tolist()
mixed_raw_feature_columns = raw_feature_columns + MIXED_ROUTE_FEATURES
full_long = pd.read_csv(require_file(FULL_LONG_TABLE_PATH))
waste_by_predictions = pd.read_csv(require_file(WASTE_BY_PREDICTIONS_PATH))
full_long = add_waste_route_features(full_long)
missing_mixed_features = [column for column in mixed_raw_feature_columns if column not in full_long.columns]
if missing_mixed_features:
    raise RuntimeError(f"Missing mixed feature columns: {missing_mixed_features}")
full_long[mixed_raw_feature_columns] = full_long[mixed_raw_feature_columns].apply(pd.to_numeric, errors="coerce")
if int(full_long[mixed_raw_feature_columns].isna().sum().sum()) != 0:
    raise RuntimeError("Mixed feature contract contains NaN values after numeric coercion")
full_long[TARGET_COL] = np.where(
    full_long["Waste_Generation_t"].notna(),
    np.log1p(pd.to_numeric(full_long["Waste_Generation_t"], errors="coerce")),
    np.nan,
)
mixed_train = full_long[full_long["Waste_Generation_t"].notna()].copy().reset_index(drop=True)
print({"step": "mixed_inputs_ready", "train_rows": int(len(mixed_train)), "all_rows": int(len(full_long))})

mixed_pipeline = fit_mixed_preprocessor(mixed_train, mixed_raw_feature_columns)
mixed_train_processed = transform_with_mixed_preprocessor(mixed_pipeline, mixed_train)
mixed_processed_columns = mixed_train_processed.columns.astype(str).tolist()
mixed_all_processed = transform_with_mixed_preprocessor(mixed_pipeline, full_long)
pd.DataFrame({"feature": mixed_raw_feature_columns}).to_csv(MIXED_RAW_CONTRACT_PATH, index=False)
pd.DataFrame({"feature": mixed_processed_columns}).to_csv(MIXED_PROCESSED_CONTRACT_PATH, index=False)
with open(MIXED_PIPELINE_PATH, "wb") as handle:
    pickle.dump(mixed_pipeline, handle)
print({"step": "mixed_pipeline_ready", "processed_features": len(mixed_processed_columns)})


In [ ]:
mixed_processed_columns

In [ ]:
model_version = resolve_model_version()
mixed_model = TabPFNRegressor.create_default_for_version(
    model_version,
    n_estimators=TABPFN_N_ESTIMATORS,
    device="cuda" if torch.cuda.is_available() else "cpu",
    fit_mode="low_memory",
    memory_saving_mode="auto",
    ignore_pretraining_limits=True,
    inference_config={"SUBSAMPLE_SAMPLES": TABPFN_SUBSAMPLE_SAMPLES},
    random_state=SEED,
)
print(f"Training mixed finalize model on {len(mixed_train_processed)} rows with {model_version.value}...")
mixed_model.fit(mixed_train_processed[mixed_processed_columns], mixed_train[TARGET_COL].to_numpy(dtype=float))
mixed_predictions = predict_in_batches(
    mixed_model,
    mixed_all_processed[mixed_processed_columns],
    batch_size=BATCH_SIZE,
    progress_every=PREDICT_PROGRESS_EVERY,
)
with open(MIXED_MODEL_PATH, "wb") as handle:
    pickle.dump(mixed_model, handle)
del mixed_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
mixed_pred_raw = np.expm1(mixed_predictions)
observed_raw = pd.to_numeric(full_long["Waste_Generation_t"], errors="coerce").to_numpy(dtype=float)
has_observed = np.isfinite(observed_raw)
mixed_output = full_long[CONTEXT_COLUMNS].copy()
mixed_output["Label_Raw_Observed"] = observed_raw
mixed_output["Label_Log_Observed"] = np.where(has_observed, np.log1p(observed_raw), np.nan)
mixed_output["Mixed_Predicted_Raw"] = mixed_pred_raw
mixed_output["Mixed_Predicted_Log"] = mixed_predictions
mixed_output["Mixed_Final_Raw"] = np.where(has_observed, observed_raw, mixed_pred_raw)
mixed_output["Mixed_Final_Log"] = np.where(has_observed, np.log1p(observed_raw), mixed_predictions)
mixed_output["Mixed_Source"] = np.where(has_observed, "observed", "predicted")
mixed_output["HasObservedLabel"] = has_observed
mixed_output = mixed_output[PREDICTION_COLUMNS]
mixed_output = mixed_output.set_index(["Country Code", "Year", "WasteFlag"]).sort_index().reset_index()
mixed_output = mixed_output[PREDICTION_COLUMNS]
mixed_output.to_csv(MIXED_PREDICTIONS_PATH, index=False)
print({"step": "mixed_predictions_written", "rows": int(len(mixed_output))})


In [ ]:
waste_by_required = [
    *CONTEXT_COLUMNS,
    "Label_Raw_Observed",
    "Label_Log_Observed",
    "Predicted_Raw",
    "Predicted_Log",
    "HasObservedLabel",
]
missing_waste_by = [column for column in waste_by_required if column not in waste_by_predictions.columns]
if missing_waste_by:
    raise RuntimeError(f"Waste-by predictions missing columns: {missing_waste_by}")
comparison_frame = mixed_output.merge(
    waste_by_predictions[waste_by_required].rename(
        columns={
            "Predicted_Raw": "WasteBy_Predicted_Raw",
            "Predicted_Log": "WasteBy_Predicted_Log",
            "Label_Raw_Observed": "WasteBy_Label_Raw_Observed",
            "Label_Log_Observed": "WasteBy_Label_Log_Observed",
            "HasObservedLabel": "WasteBy_HasObservedLabel",
        }
    ),
    on=CONTEXT_COLUMNS,
    how="inner",
    validate="one_to_one",
)
if len(comparison_frame) != len(mixed_output):
    raise RuntimeError("Mixed and waste-by predictions do not align one-to-one")
if not np.allclose(
    comparison_frame["Label_Raw_Observed"].fillna(-1.0).to_numpy(dtype=float),
    comparison_frame["WasteBy_Label_Raw_Observed"].fillna(-1.0).to_numpy(dtype=float),
    rtol=1e-9,
    atol=1e-6,
):
    raise RuntimeError("Mixed and waste-by observed labels differ")
if not comparison_frame["HasObservedLabel"].astype(bool).equals(comparison_frame["WasteBy_HasObservedLabel"].astype(bool)):
    raise RuntimeError("Mixed and waste-by observed flags differ")
route_comparison_overall, route_comparison_by_waste, high_impact_country_waste = build_route_switch_comparison(comparison_frame)
msw_contamination_diagnostics = build_msw_contamination_diagnostics(comparison_frame)
route_comparison_overall.to_csv(ROUTE_COMPARISON_OVERALL_PATH, index=False)
route_comparison_by_waste.to_csv(ROUTE_COMPARISON_BY_WASTE_PATH, index=False)
high_impact_country_waste.to_csv(HIGH_IMPACT_PATH, index=False)
msw_contamination_diagnostics.to_csv(MSW_DIAGNOSTICS_PATH, index=False)
print({"step": "route_comparison_written", "overall_rows": int(len(route_comparison_overall)), "high_impact_rows": int(len(high_impact_country_waste))})


In [ ]:
route_comparison_overall


In [ ]:
route_comparison_by_waste


In [ ]:
high_impact_country_waste.head(20)


In [ ]:
msw_contamination_diagnostics
